In [2]:
import numpy as np
from pathlib import Path
import torch

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
)


# =============================================================================
# Paths
# =============================================================================

NEURAL_PATH = "/home/maria/Science/data/hybrid_neural_responses_reduced.npy"
HUMAN_LABEL_PATH = "/home/maria/Science/data/image_labels.npy"

OUTDIR = Path("/home/maria/Science/results/human_adam_geometry_test")
OUTDIR.mkdir(parents=True, exist_ok=True)

LOO_RESULTS_PATH = OUTDIR / "adam_human_loo_with_fold_directions.npz"
FULL_DIRECTION_RESULTS_PATH = OUTDIR / "adam_human_full_direction_geometry_test.npz"


# =============================================================================
# Settings
# =============================================================================

LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 3000

N_PERMUTATIONS = 10000
RANDOM_SEED = 0
EPS = 1e-12

EXPECTED_MIN_ACCURACY = 0.72


# =============================================================================
# Data loading
# =============================================================================

def load_human_labels(path: str) -> np.ndarray:
    labels = np.load(path, allow_pickle=True).item()["labels"]
    labels = np.asarray(labels).astype(np.int64)

    print(f"Loaded human labels: {labels.shape}")
    print("Human label counts [inanimate, animate], excluding -1:")
    print(np.bincount(labels[labels != -1], minlength=2))
    print("Convention: 0 = inanimate, 1 = animate")

    return labels


def load_clean_neural_and_human_labels():
    X = np.load(NEURAL_PATH, allow_pickle=True)
    X = np.asarray(X, dtype=np.float64)

    y = load_human_labels(HUMAN_LABEL_PATH)

    print()
    print("=" * 80)
    print("Loading neural data")
    print("=" * 80)
    print(f"Raw neural shape: {X.shape}")
    print(f"Labels shape:     {y.shape}")

    if X.shape[0] != len(y) and X.shape[1] == len(y):
        print("[INFO] Transposing neural matrix to images x features.")
        X = X.T

    if X.shape[0] != len(y):
        raise ValueError(f"Expected rows to match labels. Got X={X.shape}, y={y.shape}")

    labeled_mask = y != -1
    original_indices = np.where(labeled_mask)[0]

    X = X[labeled_mask]
    y = y[labeled_mask]

    finite_cols = np.all(np.isfinite(X), axis=0)

    good_cols = np.zeros(X.shape[1], dtype=bool)
    finite_indices = np.where(finite_cols)[0]

    nonconstant_finite = np.std(X[:, finite_cols], axis=0) > 1e-12
    good_cols[finite_indices[nonconstant_finite]] = True

    X = X[:, good_cols]

    print(f"Clean neural shape: {X.shape}")
    print(f"Removed bad/nonconstant columns: {np.sum(~good_cols)}")
    print(f"Final label counts [inanimate, animate]: {np.bincount(y, minlength=2)}")

    return X, y, labeled_mask, original_indices, good_cols


# =============================================================================
# Adam logistic regression
# =============================================================================

def sigmoid_np(z):
    z = np.clip(z, -40, 40)
    return 1.0 / (1.0 + np.exp(-z))


class TorchLogisticRegression(torch.nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = torch.nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x)


def fit_adam_logistic_axis(
    X_train,
    y_train,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    epochs=EPOCHS,
    seed=0,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    X_t = torch.tensor(X_train.astype(np.float32))
    y_t = torch.tensor(y_train.astype(np.float32)).view(-1, 1)

    model = TorchLogisticRegression(X_train.shape[1])

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    loss_fn = torch.nn.BCEWithLogitsLoss()

    for _ in range(epochs):
        optimizer.zero_grad()
        logits = model(X_t)
        loss = loss_fn(logits, y_t)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        w = model.linear.weight.detach().cpu().numpy().ravel().astype(np.float64)
        b = float(model.linear.bias.detach().cpu().numpy()[0])

    return w, b


def normalize(v, eps=EPS):
    norm = np.linalg.norm(v)
    if norm < eps:
        raise ValueError("Cannot normalize near-zero vector.")
    return v / norm


# =============================================================================
# Geometry helpers
# =============================================================================

def one_vector_geometry_score(x_scaled, w):
    """
    x_scaled and w live in the same standardized coordinate system.

    Returns:
      parallel        = <x, u>
      orthogonal_norm = ||x - <x,u>u||
      angle_t         = sqrt(d - 1) * parallel / orthogonal_norm

    This is the Adam-direction version of sqrt(nu) * cot(theta).
    """
    u = normalize(w)

    parallel = float(x_scaled @ u)
    x_orth = x_scaled - parallel * u
    orthogonal_norm = float(np.linalg.norm(x_orth))

    d = x_scaled.shape[0]
    nu = d - 1

    angle_t = float(np.sqrt(nu) * parallel / (orthogonal_norm + EPS))

    return parallel, orthogonal_norm, angle_t


def batch_geometry_scores(X_scaled, w):
    u = normalize(w)

    parallel = X_scaled @ u
    X_orth = X_scaled - np.outer(parallel, u)
    orthogonal_norm = np.linalg.norm(X_orth, axis=1)

    d = X_scaled.shape[1]
    nu = d - 1

    angle_t = np.sqrt(nu) * parallel / (orthogonal_norm + EPS)

    return parallel, orthogonal_norm, angle_t, nu, u


# =============================================================================
# Statistics
# =============================================================================

def class_difference(scores, y):
    return float(scores[y == 1].mean() - scores[y == 0].mean())


def permutation_test(scores, y, n_permutations=N_PERMUTATIONS, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)

    observed = class_difference(scores, y)
    null = np.zeros(n_permutations, dtype=np.float64)

    for b in range(n_permutations):
        y_perm = rng.permutation(y)
        null[b] = class_difference(scores, y_perm)

    p_one_sided = (1.0 + np.sum(null >= observed)) / (n_permutations + 1.0)
    p_two_sided = (1.0 + np.sum(np.abs(null) >= abs(observed))) / (n_permutations + 1.0)

    return observed, null, p_one_sided, p_two_sided


def summarize_binary_predictions(y, scores, probs, preds):
    acc = accuracy_score(y, preds)
    bal_acc = balanced_accuracy_score(y, preds)
    auc = roc_auc_score(y, probs)
    cm = confusion_matrix(y, preds, labels=[0, 1])

    print()
    print("=" * 80)
    print("LOO Adam logistic regression, human labels")
    print("=" * 80)
    print(f"Accuracy:          {acc:.4f}")
    print(f"Balanced accuracy: {bal_acc:.4f}")
    print(f"AUC:               {auc:.4f}")
    print("Confusion matrix rows=true [inanimate, animate], cols=pred:")
    print(cm)

    if acc >= EXPECTED_MIN_ACCURACY:
        print(f"[OK] Accuracy is >= {EXPECTED_MIN_ACCURACY:.2f}.")
    else:
        print(f"[WARNING] Accuracy is below {EXPECTED_MIN_ACCURACY:.2f}.")

    return acc, bal_acc, auc, cm


def summarize_geometry_score(name, scores, y):
    inanimate = scores[y == 0]
    animate = scores[y == 1]

    print()
    print("=" * 80)
    print(name)
    print("=" * 80)
    print(f"Inanimate mean: {inanimate.mean():+.6f}")
    print(f"Animate mean:   {animate.mean():+.6f}")
    print(f"Animate - inanimate: {animate.mean() - inanimate.mean():+.6f}")
    print(f"Inanimate std:  {inanimate.std(ddof=1):.6f}")
    print(f"Animate std:    {animate.std(ddof=1):.6f}")

    try:
        auc = roc_auc_score(y, scores)
        print(f"AUC using score directly: {auc:.6f}")
    except Exception as e:
        print(f"AUC failed: {e}")


# =============================================================================
# LOO training
# =============================================================================

def run_loo_adam_with_geometry(X, y):
    n, d = X.shape

    loo_logits = np.zeros(n, dtype=np.float64)
    loo_probs = np.zeros(n, dtype=np.float64)
    loo_preds = np.zeros(n, dtype=np.int64)
    loo_biases = np.zeros(n, dtype=np.float64)

    loo_parallel = np.zeros(n, dtype=np.float64)
    loo_orthogonal_norm = np.zeros(n, dtype=np.float64)
    loo_angle_t = np.zeros(n, dtype=np.float64)

    # Fold directions are saved in standardized fold coordinates.
    # Shape: n folds x d features.
    fold_weights = np.zeros((n, d), dtype=np.float32)

    for test_idx in range(n):
        train_mask = np.arange(n) != test_idx

        X_train_raw = X[train_mask]
        y_train = y[train_mask]

        X_test_raw = X[~train_mask]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train_raw)
        X_test = scaler.transform(X_test_raw)

        w, b = fit_adam_logistic_axis(
            X_train,
            y_train,
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            epochs=EPOCHS,
            seed=test_idx,
        )

        logit = float(X_test[0] @ w + b)
        prob = float(sigmoid_np(logit))
        pred = int(prob >= 0.5)

        parallel, orthogonal_norm, angle_t = one_vector_geometry_score(
            X_test[0],
            w,
        )

        loo_logits[test_idx] = logit
        loo_probs[test_idx] = prob
        loo_preds[test_idx] = pred
        loo_biases[test_idx] = b

        loo_parallel[test_idx] = parallel
        loo_orthogonal_norm[test_idx] = orthogonal_norm
        loo_angle_t[test_idx] = angle_t

        fold_weights[test_idx] = w.astype(np.float32)

        print(
            f"[Human Adam LOO {test_idx + 1:03d}/{n}] "
            f"true={y[test_idx]} "
            f"logit={logit:+.6f} "
            f"prob={prob:.4f} "
            f"pred={pred} "
            f"angle_t={angle_t:+.6f}"
        )

    return {
        "loo_logits": loo_logits,
        "loo_probs": loo_probs,
        "loo_preds": loo_preds,
        "loo_biases": loo_biases,
        "loo_parallel": loo_parallel,
        "loo_orthogonal_norm": loo_orthogonal_norm,
        "loo_angle_t": loo_angle_t,
        "fold_weights_standardized": fold_weights,
    }


# =============================================================================
# Full-data direction training
# =============================================================================

def run_full_direction_geometry(X, y):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    w_full, b_full = fit_adam_logistic_axis(
        X_scaled,
        y,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        epochs=EPOCHS,
        seed=12345,
    )

    logits_full = X_scaled @ w_full + b_full
    probs_full = sigmoid_np(logits_full)
    preds_full = (probs_full >= 0.5).astype(np.int64)

    parallel, orthogonal_norm, angle_t, nu, u_full = batch_geometry_scores(
        X_scaled,
        w_full,
    )

    return {
        "scaler_mean": scaler.mean_,
        "scaler_scale": scaler.scale_,
        "w_full_standardized": w_full,
        "u_full_standardized": u_full,
        "b_full": b_full,
        "full_logits": logits_full,
        "full_probs": probs_full,
        "full_preds": preds_full,
        "full_parallel": parallel,
        "full_orthogonal_norm": orthogonal_norm,
        "full_angle_t": angle_t,
        "nu": nu,
    }


# =============================================================================
# Main
# =============================================================================

def main():
    print()
    print("#" * 80)
    print("Retrain human-label Adam decoder + animacy direction geometry test")
    print("#" * 80)

    X, y, labeled_mask, original_indices, good_cols = load_clean_neural_and_human_labels()

    print()
    print("#" * 80)
    print("Part 1: LOO Adam training")
    print("#" * 80)

    loo = run_loo_adam_with_geometry(X, y)

    acc, bal_acc, auc, cm = summarize_binary_predictions(
        y=y,
        scores=loo["loo_logits"],
        probs=loo["loo_probs"],
        preds=loo["loo_preds"],
    )

    summarize_geometry_score(
        "Cross-validated projection onto fold Adam direction",
        loo["loo_parallel"],
        y,
    )

    summarize_geometry_score(
        "Cross-validated hypersphere/t-style angle score",
        loo["loo_angle_t"],
        y,
    )

    obs_cv_angle, null_cv_angle, p1_cv_angle, p2_cv_angle = permutation_test(
        loo["loo_angle_t"],
        y,
        n_permutations=N_PERMUTATIONS,
        seed=RANDOM_SEED,
    )

    obs_cv_parallel, null_cv_parallel, p1_cv_parallel, p2_cv_parallel = permutation_test(
        loo["loo_parallel"],
        y,
        n_permutations=N_PERMUTATIONS,
        seed=RANDOM_SEED,
    )

    print()
    print("=" * 80)
    print("Permutation test on cross-validated angle scores")
    print("=" * 80)
    print(f"Observed animate - inanimate diff: {obs_cv_angle:+.6f}")
    print(f"Null mean: {null_cv_angle.mean():+.6f}")
    print(f"Null std:  {null_cv_angle.std(ddof=1):.6f}")
    print("Null 2.5%, 50%, 97.5%:")
    print(np.quantile(null_cv_angle, [0.025, 0.5, 0.975]))
    print(f"One-sided p-value, animate > inanimate: {p1_cv_angle:.8f}")
    print(f"Two-sided p-value: {p2_cv_angle:.8f}")

    np.savez_compressed(
        LOO_RESULTS_PATH,
        y=y,
        labeled_mask=labeled_mask,
        original_indices=original_indices,
        good_cols=good_cols,
        accuracy=acc,
        balanced_accuracy=bal_acc,
        auc=auc,
        confusion_matrix=cm,
        loo_logits=loo["loo_logits"],
        loo_probs=loo["loo_probs"],
        loo_preds=loo["loo_preds"],
        loo_biases=loo["loo_biases"],
        loo_parallel=loo["loo_parallel"],
        loo_orthogonal_norm=loo["loo_orthogonal_norm"],
        loo_angle_t=loo["loo_angle_t"],
        fold_weights_standardized=loo["fold_weights_standardized"],
        observed_cv_parallel_diff=obs_cv_parallel,
        null_cv_parallel_diff=null_cv_parallel,
        p_one_sided_cv_parallel=p1_cv_parallel,
        p_two_sided_cv_parallel=p2_cv_parallel,
        observed_cv_angle_t_diff=obs_cv_angle,
        null_cv_angle_t_diff=null_cv_angle,
        p_one_sided_cv_angle_t=p1_cv_angle,
        p_two_sided_cv_angle_t=p2_cv_angle,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        epochs=EPOCHS,
    )

    print()
    print("=" * 80)
    print("Saved LOO results")
    print("=" * 80)
    print(LOO_RESULTS_PATH)

    print()
    print("#" * 80)
    print("Part 2: Train one final full-data Adam animacy direction")
    print("#" * 80)

    full = run_full_direction_geometry(X, y)

    full_acc, full_bal_acc, full_auc, full_cm = summarize_binary_predictions(
        y=y,
        scores=full["full_logits"],
        probs=full["full_probs"],
        preds=full["full_preds"],
    )

    summarize_geometry_score(
        "Full-data projection onto Adam animacy direction",
        full["full_parallel"],
        y,
    )

    summarize_geometry_score(
        "Full-data hypersphere/t-style angle score",
        full["full_angle_t"],
        y,
    )

    obs_full_angle, null_full_angle, p1_full_angle, p2_full_angle = permutation_test(
        full["full_angle_t"],
        y,
        n_permutations=N_PERMUTATIONS,
        seed=RANDOM_SEED,
    )

    obs_full_parallel, null_full_parallel, p1_full_parallel, p2_full_parallel = permutation_test(
        full["full_parallel"],
        y,
        n_permutations=N_PERMUTATIONS,
        seed=RANDOM_SEED,
    )

    print()
    print("=" * 80)
    print("Permutation test on full-data Adam angle scores")
    print("=" * 80)
    print("[NOTE] This is descriptive because the direction was learned from all labels.")
    print("For honest held-out evidence, prefer the cross-validated angle score above.")
    print()
    print(f"Observed animate - inanimate diff: {obs_full_angle:+.6f}")
    print(f"Null mean: {null_full_angle.mean():+.6f}")
    print(f"Null std:  {null_full_angle.std(ddof=1):.6f}")
    print("Null 2.5%, 50%, 97.5%:")
    print(np.quantile(null_full_angle, [0.025, 0.5, 0.975]))
    print(f"One-sided p-value, animate > inanimate: {p1_full_angle:.8f}")
    print(f"Two-sided p-value: {p2_full_angle:.8f}")

    np.savez_compressed(
        FULL_DIRECTION_RESULTS_PATH,
        y=y,
        labeled_mask=labeled_mask,
        original_indices=original_indices,
        good_cols=good_cols,
        full_accuracy=full_acc,
        full_balanced_accuracy=full_bal_acc,
        full_auc=full_auc,
        full_confusion_matrix=full_cm,
        scaler_mean=full["scaler_mean"],
        scaler_scale=full["scaler_scale"],
        w_full_standardized=full["w_full_standardized"],
        u_full_standardized=full["u_full_standardized"],
        b_full=full["b_full"],
        full_logits=full["full_logits"],
        full_probs=full["full_probs"],
        full_preds=full["full_preds"],
        full_parallel=full["full_parallel"],
        full_orthogonal_norm=full["full_orthogonal_norm"],
        full_angle_t=full["full_angle_t"],
        nu=full["nu"],
        observed_full_parallel_diff=obs_full_parallel,
        null_full_parallel_diff=null_full_parallel,
        p_one_sided_full_parallel=p1_full_parallel,
        p_two_sided_full_parallel=p2_full_parallel,
        observed_full_angle_t_diff=obs_full_angle,
        null_full_angle_t_diff=null_full_angle,
        p_one_sided_full_angle_t=p1_full_angle,
        p_two_sided_full_angle_t=p2_full_angle,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        epochs=EPOCHS,
    )

    print()
    print("=" * 80)
    print("Saved full-data direction geometry results")
    print("=" * 80)
    print(FULL_DIRECTION_RESULTS_PATH)

    print()
    print("Done.")


if __name__ == "__main__":
    main()


################################################################################
Retrain human-label Adam decoder + animacy direction geometry test
################################################################################
Loaded human labels: (118,)
Human label counts [inanimate, animate], excluding -1:
[62 56]
Convention: 0 = inanimate, 1 = animate

Loading neural data
Raw neural shape: (39209, 118)
Labels shape:     (118,)
[INFO] Transposing neural matrix to images x features.
Clean neural shape: (118, 39209)
Removed bad/nonconstant columns: 0
Final label counts [inanimate, animate]: [62 56]

################################################################################
Part 1: LOO Adam training
################################################################################
[Human Adam LOO 001/118] true=1 logit=-2.554781 prob=0.0721 pred=0 angle_t=-3.066589
[Human Adam LOO 002/118] true=1 logit=-1.164098 prob=0.2379 pred=0 angle_t=-1.164101
[Human Adam LOO 003/118] true=1 